# Lab 04: Build and Run Pipeline Jobs

> **Source:** Microsoft Learning &ndash; [mslearn-mlops](https://github.com/MicrosoftLearning/mslearn-mlops)
> **Original:** `experimentation/Pipelines.ipynb` | **License:** MIT

## Learning Objectives

After completing this lab you will be able to:

1. **Define pipeline components** in YAML with typed inputs and outputs.
2. **Build a pipeline** using the `@pipeline` decorator from `azure.ai.ml.dsl`.
3. **Connect components** so that inputs and outputs form a directed acyclic graph (DAG).
4. **Submit and monitor** a pipeline job in Azure ML Studio.

| | |
|---|---|
| **Estimated time** | ~20 minutes |
| **Estimated cost** | ~$0.50 |

> **Exam Tip:** Understand the data asset types that flow between components:
> - `uri_file` -- a single file (e.g., a CSV)
> - `uri_folder` -- a directory of files
> - `mlflow_model` -- an MLflow-packaged model artifact
>
> Output modes (`upload`, `rw_mount`, `ro_mount`) control *how* the data is written. `upload` copies data to the default datastore after the step completes.

## Architecture

```mermaid
graph LR
    A["diabetes-data<br/>(uri_file)"] --> B["prep_data"]
    B --> C["cleaned data<br/>(uri_folder)"]
    C --> D["train_model"]
    D --> E["trained model<br/>(mlflow_model)"]
```

The pipeline has two components:

| Step | Input Type | Output Type | Description |
|------|-----------|-------------|-------------|
| **prep_data** | `uri_file` | `uri_folder` | Drop nulls, scale numeric columns |
| **train_model** | `uri_folder` | `mlflow_model` | Train logistic regression, log metrics, save model |

In [ ]:
# --- Prerequisites: connect to Azure ML workspace ---
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except Exception:
    credential = InteractiveBrowserCredential()

ml_client = MLClient.from_config(credential=credential)
print(
    f"Connected to workspace '{ml_client.workspace_name}' "
    f"in resource group '{ml_client.resource_group_name}'"
)

## Pipeline Components

This pipeline uses two components, each defined by a **YAML specification** in the `scripts/` folder:

| File | Component | Description |
|---|---|---|
| `scripts/prep-data.yml` | `prep_data` | Reads raw CSV, drops nulls, scales numeric columns with MinMaxScaler |
| `scripts/train-model.yml` | `train_model` | Trains a LogisticRegression model, logs Accuracy and AUC, saves an MLflow model |

Each YAML file declares `inputs`, `outputs`, `code`, `environment`, and the `command` to run. This makes components **reusable** across pipelines.

In [ ]:
from azure.ai.ml import MLClient, Input, load_component
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.dsl import pipeline
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
import os

# Load component definitions from YAML
prep_data = load_component(source="scripts/prep-data.yml")
train_model = load_component(source="scripts/train-model.yml")
print(f"Components loaded: {prep_data.name}, {train_model.name}")

### Build the pipeline with @pipeline decorator

The `@pipeline` decorator from `azure.ai.ml.dsl` turns a Python function into a pipeline definition. Inside the function you:

1. Call each loaded component as a function, passing inputs.
2. Wire outputs of one step to inputs of the next.
3. Return a dictionary mapping output names to component outputs.

> **Exam Tip:** `load_component()` reads a YAML component spec. The `@pipeline` decorator (from `azure.ai.ml.dsl`) builds the DAG. These are the two key constructs for Azure ML pipelines with the Python SDK v2.

In [ ]:
@pipeline()
def diabetes_classification(pipeline_job_input):
    """Two-step pipeline: prep data -> train model."""
    clean_data = prep_data(input_data=pipeline_job_input)
    train = train_model(training_data=clean_data.outputs.output_data)
    return {
        "pipeline_job_transformed_data": clean_data.outputs.output_data,
        "pipeline_job_trained_model": train.outputs.model_output,
    }

# Instantiate the pipeline with input data
pipeline_job = diabetes_classification(
    Input(type=AssetTypes.URI_FILE, path="azureml:diabetes-data:1")
)

# Configure output modes and compute
pipeline_job.outputs.pipeline_job_transformed_data.mode = "upload"
pipeline_job.outputs.pipeline_job_trained_model.mode = "upload"
pipeline_job.settings.default_compute = "cc-ai300"
pipeline_job.settings.default_datastore = "workspaceblobstore"

# Submit the pipeline job
pipeline_job = ml_client.jobs.create_or_update(
    pipeline_job, experiment_name="pipeline_diabetes"
)
print(f"Pipeline submitted: {pipeline_job.name}")
print(f"Monitor at: {pipeline_job.studio_url}")

## Key Takeaways

- **Pipelines are DAGs** -- each component is a node, and typed inputs/outputs define the edges.
- **YAML component definitions** declare inputs, outputs, code, environment, and command -- making components portable and reusable.
- The **`@pipeline` decorator** (from `azure.ai.ml.dsl`) turns a Python function into a pipeline definition. Inside, you call components and wire their outputs to downstream inputs.
- **Typed connections** (`uri_file`, `uri_folder`, `mlflow_model`) ensure data flows correctly between steps.
- Components are **reusable** -- the same `prep_data` or `train_model` component can appear in multiple pipelines without modification.